# Notebook 3 of 3 — Evaluation with Kaggle models (manual §5.4 step 6, Task 7)

Create this notebook as a **utility benchmark** (§13.1) from https://www.kaggle.com/benchmarks/tasks/new so that
`kaggle_benchmarks` and the free model proxy are available. **No GPU needed**, Internet **On**.
**Input:** Add Input → the saved version of **Notebook 2 (generation)**.

Judging reads only the saved pairwise JSONL files; no model outputs are regenerated (§13.1.4).
Reliability (§13.2): 5 pairs per request, ≥5 s sleep between attempts, up to 3 retries per batch, failures logged
to `*_failures.jsonl`, judgments checkpointed after every batch, `--reuse_judgments` resumes, and win / loss /
tie / unknown / failure rates are summarized.

In [ ]:
# Manual section 7.1: clone the repository in Kaggle (Notebook settings: Internet ON).
import os
REPO_URL = "https://github.com/mahamasiff/cs5326-PA2-27100015.git"
REPO_DIR = "/kaggle/working/cs5326-PA2-27100015"
if not os.path.exists(REPO_DIR):
    !git clone -q {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} pull -q
%cd {REPO_DIR}
!git log --oneline -1

In [ ]:
import subprocess
def sh(cmd):
    """Run a shell command, stream its output, and stop the notebook if it fails."""
    print("$", cmd, flush=True)
    result = subprocess.run(cmd, shell=True)
    if result.returncode != 0:
        raise RuntimeError(f"command failed with exit code {result.returncode}: {cmd}")

In [ ]:
import kaggle_benchmarks as kbench
JUDGE = "google/gemini-3.6-flash"   # §13: if unavailable, choose another listed model and record it in the report
print(sorted(kbench.llms))

In [ ]:
# Locate files from the attached input dataset (Add Input -> the saved version of the previous notebook).
import glob
def find_input(pattern):
    hits = sorted(glob.glob(f"/kaggle/input/**/{pattern}", recursive=True))
    assert hits, f"Attach the saved notebook output that contains {pattern}"
    print(pattern, "->", hits[0])
    return hits[0]

In [ ]:
import shutil
os.makedirs("outputs/eval", exist_ok=True)
for name in ("dpo_uf_vs_sft_pairs.jsonl", "dpo_hh_vs_sft_pairs.jsonl", "lora_vs_base_pairs.jsonl"):
    shutil.copy(find_input(f"eval/{name}"), f"outputs/eval/{name}")
# Resume support: copy judgments from a previous version of this notebook if it is attached too.
for pattern in ("eval/*_judged.jsonl", "eval/*_judged_failures.jsonl"):
    for path in glob.glob(f"/kaggle/input/**/{pattern}", recursive=True):
        shutil.copy(path, "outputs/eval/")
!ls -l outputs/eval

In [ ]:
# Smoke run on 2 pairs to confirm the judge responds and its output parses.
sh("python scripts/judge_pairwise_kaggle.py --pairs_path outputs/eval/dpo_uf_vs_sft_pairs.jsonl "
   f"--output_path outputs/eval/smoke_judged.jsonl --judge_model {JUDGE} --limit 2 "
   "--batch_size 5 --sleep_seconds 5 --max_retries 3")
!head -c 2000 outputs/eval/smoke_judged_raw.jsonl
!rm -f outputs/eval/smoke_judged*

## DPO vs SFT (both tracks) and LoRA-DPO vs base

In [ ]:
for name in ("dpo_uf_vs_sft", "dpo_hh_vs_sft", "lora_vs_base"):
    sh(f"python scripts/judge_pairwise_kaggle.py --pairs_path outputs/eval/{name}_pairs.jsonl "
       f"--output_path outputs/eval/{name}_judged.jsonl --judge_model {JUDGE} "
       "--batch_size 5 --sleep_seconds 5 --max_retries 3 --reuse_judgments")

In [ ]:
sh("python scripts/collect_results.py --eval_dir outputs/eval --output outputs/eval/results.md")
!cd outputs && zip -q -r /kaggle/working/evaluation_outputs.zip eval
!ls -lh /kaggle/working/evaluation_outputs.zip

**Save Version** when judging completes. If some batches failed, re-run the judging cell: completed pairs are reused.